# 02 — Train Model
**MyVoterWisdom · [github.com/sysWisdom/myvoterwisdom](https://github.com/sysWisdom/myvoterwisdom)**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sysWisdom/myvoterwisdom/blob/main/notebooks/02_train_model.ipynb)

> Non-partisan educational tool. See [DISCLAIMER.md](../DISCLAIMER.md) before use.

This notebook walks through the full training pipeline from `train_model.py` step-by-step:
preprocessing → feature engineering → Random Forest training → evaluation.

## Step 1 — Environment Setup

In [ ]:
import os, sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('/content/myvoterwisdom'):
        os.system('git clone https://github.com/sysWisdom/myvoterwisdom.git /content/myvoterwisdom')
    REPO_ROOT = '/content/myvoterwisdom'
    os.system('pip install -q imbalanced-learn')
else:
    REPO_ROOT = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..'))

sys.path.insert(0, REPO_ROOT)
DATA_PATH  = os.path.join(REPO_ROOT, 'data', 'voting_pres_data.csv')
MODEL_PATH = os.path.join(REPO_ROOT, 'model', 'wisdom_rf.joblib')
os.makedirs(os.path.join(REPO_ROOT, 'model'), exist_ok=True)
print(f"REPO_ROOT : {REPO_ROOT}")

## Step 2 — Load Data & Run Preprocessing Pipeline

The preprocessing pipeline (`preprocess.py`) adds three comparison columns then
sets `Wisdom=True` when a county meets ≥ 2 of 3 historical turnout conditions.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from preprocess import add_filter_columns, compare_votes_and_ballots, update_wisdom, prepare_features_and_target

sns.set_theme(style='whitegrid')

df = pd.read_csv(DATA_PATH)
df = add_filter_columns(df)
df = compare_votes_and_ballots(df)
df = update_wisdom(df)

print(f"Dataset shape after preprocessing: {df.shape}")
print(f"\nWisdom distribution:\n{df['Wisdom'].value_counts()}")
print("\n⚠️  If all values are False, the model has a single-class problem.")
print("   See DISCLAIMER.md — this is a known limitation of the current dataset.")
df[['County', 'State', 'Election Year', 'Wisdom']].tail(10)

## Step 3 — Prepare Features & Check Class Balance

In [ ]:
from sklearn.preprocessing import OneHotEncoder
import numpy as np

X, y = prepare_features_and_target(df)
print(f"Features shape : {X.shape}")
print(f"Target shape   : {y.shape}")
print(f"Feature columns: {X.columns.tolist()}")
print(f"\nClass counts:\n{y.value_counts()}")

# Encode any categorical columns
categorical_columns = X.select_dtypes(include=['object']).columns
if len(categorical_columns):
    encoder = OneHotEncoder(sparse_output=False, drop='first')
    X_enc = pd.DataFrame(
        encoder.fit_transform(X[categorical_columns]),
        columns=encoder.get_feature_names_out(categorical_columns),
        index=X.index
    )
    X = pd.concat([X.drop(columns=categorical_columns), X_enc], axis=1)

print(f"\nFinal feature matrix shape: {X.shape}")
X.head()

## Step 4 — Train / Test Split & Random Forest Training

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import joblib

n_classes = y.nunique()
if n_classes < 2:
    print("⚠️  Only one class present in target — cannot perform stratified split or train a meaningful model.")
    print("    Add more county data with diverse Wisdom values and re-run.")
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    print(f"Train size: {len(X_train)} | Test size: {len(X_test)}")

    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)

    joblib.dump(model, MODEL_PATH)
    print(f"\nModel saved to: {MODEL_PATH}")

    y_pred = model.predict(X_test)
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, zero_division=0))

## Step 5 — Feature Importance

In [ ]:
if n_classes >= 2:
    importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=True)
    fig, ax = plt.subplots(figsize=(8, max(4, len(importances) * 0.4)))
    importances.plot(kind='barh', ax=ax, color=sns.color_palette('muted')[0])
    ax.set_title('Random Forest — Feature Importance')
    ax.set_xlabel('Importance Score')
    plt.tight_layout()
    plt.show()
else:
    print("Skipped — model was not trained (single-class problem).")